# 06 Final Evidence Pack Export


The purpose of this notebook is to:

1. Create clean final-output folders.
2. Copy only report-ready tables and figures from Notebooks 02 to 05.
3. Separate main report figures from diagnostic or sensitivity outputs.
4. Create a master summary table.
5. Create table and figure inventories.
6. Create a final evidence-pack README and manifest.



## 0. Setup

This cell detects whether the notebook is run from the project root or from the `notebooks/` folder.


In [1]:
from pathlib import Path
import shutil
import json
from datetime import datetime

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# Detect project directory whether notebook is run from project root or /notebooks
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

outputs_dir = project_dir / "outputs"
tables_dir = outputs_dir / "tables"
figures_dir = outputs_dir / "figures"

final_tables_dir = outputs_dir / "final_tables"
final_figures_dir = outputs_dir / "final_figures"
final_figures_main_dir = final_figures_dir / "main_report"
final_figures_diagnostics_dir = final_figures_dir / "diagnostics_appendix"
report_assets_dir = outputs_dir / "report_assets"

for folder in [
    final_tables_dir,
    final_figures_main_dir,
    final_figures_diagnostics_dir,
    report_assets_dir
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Current directory:", current_dir)
print("Project directory:", project_dir)
print("Tables source directory:", tables_dir)
print("Figures source directory:", figures_dir)
print("Final tables directory:", final_tables_dir)
print("Final figures directory:", final_figures_dir)
print("Report assets directory:", report_assets_dir)


Current directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/notebooks
Project directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost
Tables source directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/tables
Figures source directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/figures
Final tables directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/final_tables
Final figures directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/final_figures
Report assets directory: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_

## 1. Define final-report tables


Tables marked as `main_report` are suitable for the main report. Tables marked as `technical_appendix` are useful for transparency but should not dominate the final write-up.


In [2]:
final_table_specs = [
    # Notebook 02: validation and cohort readiness
    {
        "source_file": "notebook_02_validation_summary.csv",
        "final_file": "01_notebook_02_validation_summary.csv",
        "source_notebook": "02",
        "role": "technical_appendix",
        "purpose": "Documents raw file loaded, row counts and data validation metrics."
    },
    {
        "source_file": "notebook_02_cohort_size_assessment.csv",
        "final_file": "02_notebook_02_cohort_size_assessment.csv",
        "source_notebook": "02",
        "role": "main_report",
        "purpose": "Confirms whether the 50,000-row extract is suitable for final modelling."
    },

    # Notebook 03: descriptive cohort profile
    {
        "source_file": "notebook_03_report_summary.csv",
        "final_file": "03_notebook_03_report_summary.csv",
        "source_notebook": "03",
        "role": "main_report",
        "purpose": "Summarises primary cancer cohort size and descriptive profile."
    },
    {
        "source_file": "notebook_03_primary_vs_non_cancer_summary.csv",
        "final_file": "04_notebook_03_primary_vs_non_cancer_summary.csv",
        "source_notebook": "03",
        "role": "main_report",
        "purpose": "Compares primary cancer and non-cancer admissions."
    },

    # Notebook 04: utilisation and cost analysis
    {
        "source_file": "notebook_04_report_summary.csv",
        "final_file": "05_notebook_04_report_summary.csv",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Summarises utilisation, cost and high-cost thresholds."
    },
    {
        "source_file": "notebook_04_overall_utilisation_cost_summary.csv",
        "final_file": "06_notebook_04_overall_utilisation_cost_summary.csv",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Overall utilisation and cost summary for primary cancer admissions."
    },
    {
        "source_file": "notebook_04_utilisation_cost_by_diagnosis_group.csv",
        "final_file": "07_notebook_04_utilisation_cost_by_diagnosis_group.csv",
        "source_notebook": "04",
        "role": "technical_appendix",
        "purpose": "Diagnosis-group utilisation and cost summary. Use report-facing n≥20 interpretation only."
    },
    {
        "source_file": "notebook_04_utilisation_cost_by_primary_payer.csv",
        "final_file": "08_notebook_04_utilisation_cost_by_primary_payer.csv",
        "source_notebook": "04",
        "role": "technical_appendix",
        "purpose": "Primary payer descriptive summary. Use report-facing n≥20 interpretation only."
    },

    # Notebook 05: final modelling
    {
        "source_file": "notebook_05_report_summary.csv",
        "final_file": "09_notebook_05_report_summary.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Compact final modelling summary."
    },
    {
        "source_file": "notebook_05_model_comparison_summary.csv",
        "final_file": "10_notebook_05_model_comparison_summary.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Explains final model selection and diagnostic roles."
    },
    {
        "source_file": "notebook_05_los_negative_binomial_model_results_report_facing.csv",
        "final_file": "11_notebook_05_los_negative_binomial_model_results_report_facing.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final LOS model results excluding intercept."
    },
    {
        "source_file": "notebook_05_cost_gamma_model_results_report_facing.csv",
        "final_file": "12_notebook_05_cost_gamma_model_results_report_facing.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final cost model results excluding intercept."
    },
    {
        "source_file": "notebook_05_high_cost_logistic_model_results_report_facing.csv",
        "final_file": "13_notebook_05_high_cost_logistic_model_results_report_facing.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final high-cost admission model results excluding intercept."
    },
    {
        "source_file": "notebook_05_model_dataset_summary.csv",
        "final_file": "14_notebook_05_model_dataset_summary.csv",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Summarises model input sample and key outcome distributions."
    },
    {
        "source_file": "notebook_05_los_model_diagnostics.csv",
        "final_file": "15_notebook_05_los_model_diagnostics.csv",
        "source_notebook": "05",
        "role": "technical_appendix",
        "purpose": "Justifies the shift from Poisson to Negative Binomial LOS model."
    },
    {
        "source_file": "notebook_05_cost_model_prediction_summary.csv",
        "final_file": "16_notebook_05_cost_model_prediction_summary.csv",
        "source_notebook": "05",
        "role": "technical_appendix",
        "purpose": "Diagnostic summary for observed versus predicted total costs."
    }
]

table_inventory = pd.DataFrame(final_table_specs)
table_inventory


,source_file,final_file,source_notebook,role,purpose
0,notebook_02_validation_summary.csv,01_notebook_02_validation_summary.csv,02,technical_appendix,"Documents raw file loaded, row counts and data validation metrics."
1,notebook_02_cohort_size_assessment.csv,02_notebook_02_cohort_size_assessment.csv,02,main_report,"Confirms whether the 50,000-row extract is suitable for final modelling."
2,notebook_03_report_summary.csv,03_notebook_03_report_summary.csv,03,main_report,Summarises primary cancer cohort size and descriptive profile.
3,notebook_03_primary_vs_non_cancer_summary.csv,04_notebook_03_primary_vs_non_cancer_summary.csv,03,main_report,Compares primary cancer and non-cancer admissions.
4,notebook_04_report_summary.csv,05_notebook_04_report_summary.csv,04,main_report,"Summarises utilisation, cost and high-cost thresholds."
5,notebook_04_overall_utilisation_cost_summary.csv,06_notebook_04_overall_utilisation_cost_summary.csv,04,main_report,Overall utilisation and cost summary for primary cancer admissions.
6,notebook_04_utilisation_cost_by_diagnosis_group.csv,07_notebook_04_utilisation_cost_by_diagnosis_group.csv,04,technical_appendix,Diagnosis-group utilisation and cost summary. Use report-facing n≥20 interpretation only.
7,notebook_04_utilisation_cost_by_primary_payer.csv,08_notebook_04_utilisation_cost_by_primary_payer.csv,04,technical_appendix,Primary payer descriptive summary. Use report-facing n≥20 interpretation only.
8,notebook_05_report_summary.csv,09_notebook_05_report_summary.csv,05,main_report,Compact final modelling summary.
9,notebook_05_model_comparison_summary.csv,10_notebook_05_model_comparison_summary.csv,05,main_report,Explains final model selection and diagnostic roles.


## 2. Copy final-report tables

Missing files are flagged instead of silently ignored. If any main-report table is missing, rerun the relevant earlier notebook before writing the report.


In [3]:
copied_table_records = []

for spec in final_table_specs:
    source_path = tables_dir / spec["source_file"]
    destination_path = final_tables_dir / spec["final_file"]

    if source_path.exists():
        shutil.copy2(source_path, destination_path)
        status = "copied"
    else:
        status = "missing"

    copied_table_records.append({
        **spec,
        "source_path": str(source_path),
        "destination_path": str(destination_path),
        "status": status
    })

table_inventory = pd.DataFrame(copied_table_records)
table_inventory.to_csv(report_assets_dir / "table_inventory.csv", index=False)

display(table_inventory)

missing_main_tables = table_inventory[
    (table_inventory["status"] == "missing") &
    (table_inventory["role"] == "main_report")
]

if len(missing_main_tables) > 0:
    print("WARNING: Missing main-report tables. Rerun the relevant notebook before writing the report.")
    display(missing_main_tables)
else:
    print("All main-report tables were copied successfully.")


,source_file,final_file,source_notebook,role,purpose,source_path,destination_path,status
0,notebook_02_validation_summary.csv,01_notebook_02_validation_summary.csv,02,technical_appendix,"Documents raw file loaded, row counts and data validation metrics.",/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
1,notebook_02_cohort_size_assessment.csv,02_notebook_02_cohort_size_assessment.csv,02,main_report,"Confirms whether the 50,000-row extract is suitable for final modelling.",/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
2,notebook_03_report_summary.csv,03_notebook_03_report_summary.csv,03,main_report,Summarises primary cancer cohort size and descriptive profile.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
3,notebook_03_primary_vs_non_cancer_summary.csv,04_notebook_03_primary_vs_non_cancer_summary.csv,03,main_report,Compares primary cancer and non-cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
4,notebook_04_report_summary.csv,05_notebook_04_report_summary.csv,04,main_report,"Summarises utilisation, cost and high-cost thresholds.",/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
5,notebook_04_overall_utilisation_cost_summary.csv,06_notebook_04_overall_utilisation_cost_summary.csv,04,main_report,Overall utilisation and cost summary for primary cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
6,notebook_04_utilisation_cost_by_diagnosis_group.csv,07_notebook_04_utilisation_cost_by_diagnosis_group.csv,04,technical_appendix,Diagnosis-group utilisation and cost summary. Use report-facing n≥20 interpretation only.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
7,notebook_04_utilisation_cost_by_primary_payer.csv,08_notebook_04_utilisation_cost_by_primary_payer.csv,04,technical_appendix,Primary payer descriptive summary. Use report-facing n≥20 interpretation only.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
8,notebook_05_report_summary.csv,09_notebook_05_report_summary.csv,05,main_report,Compact final modelling summary.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
9,notebook_05_model_comparison_summary.csv,10_notebook_05_model_comparison_summary.csv,05,main_report,Explains final model selection and diagnostic roles.,/Users/marissa/Desk

All main-report tables were copied successfully.


## 3. Define final-report figures

The main report should use only selected figures. Diagnostic and sensitivity figures are retained separately for transparency.

Deprecated prototype duplicate figures are listed as `exclude` and are not copied.


In [4]:
final_figure_specs = [
    # Main descriptive / utilisation figures from Notebooks 03 and 04
    {
        "source_file": "notebook_03_median_costs_primary_vs_non_cancer.png",
        "final_file": "01_median_total_costs_primary_cancer_vs_non_cancer.png",
        "source_notebook": "03",
        "role": "main_report",
        "purpose": "Compares median total costs between primary cancer and non-cancer admissions."
    },
    {
        "source_file": "notebook_03_median_los_primary_vs_non_cancer.png",
        "final_file": "02_median_los_primary_cancer_vs_non_cancer.png",
        "source_notebook": "03",
        "role": "main_report",
        "purpose": "Compares median length of stay between primary cancer and non-cancer admissions."
    },
    {
        "source_file": "notebook_03_top_primary_cancer_diagnosis_groups.png",
        "final_file": "03_top_primary_cancer_diagnosis_groups.png",
        "source_notebook": "03",
        "role": "main_report",
        "purpose": "Describes the distribution of primary cancer diagnosis groups."
    },
    {
        "source_file": "notebook_04_median_costs_by_severity.png",
        "final_file": "04_median_total_costs_by_severity.png",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Shows unadjusted cost gradient by APR severity group."
    },
    {
        "source_file": "notebook_04_median_los_by_severity.png",
        "final_file": "05_median_los_by_severity.png",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Shows unadjusted LOS gradient by APR severity group."
    },
    {
        "source_file": "notebook_04_median_costs_by_diagnosis_group_min_n20.png",
        "final_file": "06_median_total_costs_by_diagnosis_group_min_n20.png",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Diagnosis-group cost ranking using minimum n≥20 threshold."
    },
    {
        "source_file": "notebook_04_median_los_by_diagnosis_group_min_n20.png",
        "final_file": "07_median_los_by_diagnosis_group_min_n20.png",
        "source_notebook": "04",
        "role": "main_report",
        "purpose": "Diagnosis-group LOS ranking using minimum n≥20 threshold."
    },

    # Main final modelling figures from Notebook 05
    {
        "source_file": "notebook_05_los_negative_binomial_model_incidence_rate_ratios.png",
        "final_file": "08_final_negative_binomial_los_model_irr.png",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final adjusted LOS model using Negative Binomial GLM."
    },
    {
        "source_file": "notebook_05_cost_gamma_model_cost_ratios.png",
        "final_file": "09_final_gamma_cost_model_cost_ratios.png",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final adjusted total cost model using Gamma GLM."
    },
    {
        "source_file": "notebook_05_high_cost_model_odds_ratios.png",
        "final_file": "10_final_high_cost_logistic_model_odds_ratios.png",
        "source_notebook": "05",
        "role": "main_report",
        "purpose": "Final adjusted high-cost admission model using logistic GLM."
    },

    # Diagnostics / sensitivity figures
    {
        "source_file": "notebook_05_los_poisson_model_incidence_rate_ratios.png",
        "final_file": "A1_baseline_poisson_los_model_irr.png",
        "source_notebook": "05",
        "role": "diagnostics_appendix",
        "purpose": "Poisson LOS baseline model, retained only as diagnostic because of overdispersion."
    },
    {
        "source_file": "notebook_05_cost_log_ols_model_cost_ratios.png",
        "final_file": "A2_sensitivity_log_ols_cost_model_cost_ratios.png",
        "source_notebook": "05",
        "role": "diagnostics_appendix",
        "purpose": "Sensitivity log-linear OLS cost model."
    },
    {
        "source_file": "notebook_05_observed_vs_predicted_costs.png",
        "final_file": "A3_observed_vs_predicted_total_costs_diagnostic.png",
        "source_notebook": "05",
        "role": "diagnostics_appendix",
        "purpose": "Diagnostic plot showing observed versus predicted total costs."
    },

    # Deprecated duplicate figures, not copied
    {
        "source_file": "notebook_05_cost_model_cost_ratios.png",
        "final_file": "",
        "source_notebook": "05",
        "role": "exclude",
        "purpose": "Deprecated duplicate old cost figure. Do not use."
    },
    {
        "source_file": "notebook_05_los_model_incidence_rate_ratios.png",
        "final_file": "",
        "source_notebook": "05",
        "role": "exclude",
        "purpose": "Deprecated duplicate old LOS figure. Do not use."
    }
]

figure_inventory_seed = pd.DataFrame(final_figure_specs)
figure_inventory_seed


,source_file,final_file,source_notebook,role,purpose
0,notebook_03_median_costs_primary_vs_non_cancer.png,01_median_total_costs_primary_cancer_vs_non_cancer.png,03,main_report,Compares median total costs between primary cancer and non-cancer admissions.
1,notebook_03_median_los_primary_vs_non_cancer.png,02_median_los_primary_cancer_vs_non_cancer.png,03,main_report,Compares median length of stay between primary cancer and non-cancer admissions.
2,notebook_03_top_primary_cancer_diagnosis_groups.png,03_top_primary_cancer_diagnosis_groups.png,03,main_report,Describes the distribution of primary cancer diagnosis groups.
3,notebook_04_median_costs_by_severity.png,04_median_total_costs_by_severity.png,04,main_report,Shows unadjusted cost gradient by APR severity group.
4,notebook_04_median_los_by_severity.png,05_median_los_by_severity.png,04,main_report,Shows unadjusted LOS gradient by APR severity group.
5,notebook_04_median_costs_by_diagnosis_group_min_n20.png,06_median_total_costs_by_diagnosis_group_min_n20.png,04,main_report,Diagnosis-group cost ranking using minimum n≥20 threshold.
6,notebook_04_median_los_by_diagnosis_group_min_n20.png,07_median_los_by_diagnosis_group_min_n20.png,04,main_report,Diagnosis-group LOS ranking using minimum n≥20 threshold.
7,notebook_05_los_negative_binomial_model_incidence_rate_ratios.png,08_final_negative_binomial_los_model_irr.png,05,main_report,Final adjusted LOS model using Negative Binomial GLM.
8,notebook_05_cost_gamma_model_cost_ratios.png,09_final_gamma_cost_model_cost_ratios.png,05,main_report,Final adjusted total cost model using Gamma GLM.
9,notebook_05_high_cost_model_odds_ratios.png,10_final_high_cost_logistic_model_odds_ratios.png,05,main_report,Final adjusted high-cost admission model using logistic GLM.


## 4. Copy final-report figures

Main-report figures are copied into `outputs/final_figures/main_report/`.

Diagnostic and sensitivity figures are copied into `outputs/final_figures/diagnostics_appendix/`.

Excluded prototype duplicates are not copied.


In [5]:
copied_figure_records = []

for spec in final_figure_specs:
    source_path = figures_dir / spec["source_file"]

    if spec["role"] == "main_report":
        destination_path = final_figures_main_dir / spec["final_file"]
    elif spec["role"] == "diagnostics_appendix":
        destination_path = final_figures_diagnostics_dir / spec["final_file"]
    else:
        destination_path = None

    if spec["role"] == "exclude":
        status = "excluded"
    elif source_path.exists():
        shutil.copy2(source_path, destination_path)
        status = "copied"
    else:
        status = "missing"

    copied_figure_records.append({
        **spec,
        "source_path": str(source_path),
        "destination_path": str(destination_path) if destination_path is not None else "",
        "status": status
    })

figure_inventory = pd.DataFrame(copied_figure_records)
figure_inventory.to_csv(report_assets_dir / "figure_inventory.csv", index=False)

display(figure_inventory)

missing_main_figures = figure_inventory[
    (figure_inventory["status"] == "missing") &
    (figure_inventory["role"] == "main_report")
]

if len(missing_main_figures) > 0:
    print("WARNING: Missing main-report figures. Rerun the relevant notebook or check figure filenames.")
    display(missing_main_figures)
else:
    print("All main-report figures were copied successfully.")


,source_file,final_file,source_notebook,role,purpose,source_path,destination_path,status
0,notebook_03_median_costs_primary_vs_non_cancer.png,01_median_total_costs_primary_cancer_vs_non_cancer.png,03,main_report,Compares median total costs between primary cancer and non-cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
1,notebook_03_median_los_primary_vs_non_cancer.png,02_median_los_primary_cancer_vs_non_cancer.png,03,main_report,Compares median length of stay between primary cancer and non-cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
2,notebook_03_top_primary_cancer_diagnosis_groups.png,03_top_primary_cancer_diagnosis_groups.png,03,main_report,Describes the distribution of primary cancer diagnosis groups.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
3,notebook_04_median_costs_by_severity.png,04_median_total_costs_by_severity.png,04,main_report,Shows unadjusted cost gradient by APR severity group.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
4,notebook_04_median_los_by_severity.png,05_median_los_by_severity.png,04,main_report,Shows unadjusted LOS gradient by APR severity group.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
5,notebook_04_median_costs_by_diagnosis_group_min_n20.png,06_median_total_costs_by_diagnosis_group_min_n20.png,04,main_report,Diagnosis-group cost ranking using minimum n≥20 threshold.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
6,notebook_04_median_los_by_diagnosis_group_min_n20.png,07_median_los_by_diagnosis_group_min_n20.png,04,main_report,Diagnosis-group LOS ranking using minimum n≥20 threshold.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
7,notebook_05_los_negative_binomial_model_incidence_rate_ratios.png,08_final_negative_binomial_los_model_irr.png,05,main_report,Final adjusted LOS model using Negative Binomial GLM.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
8,notebook_05_cost_gamma_model_cost_ratios.png,09_final_gamma_cost_model_cost_ratios.png,05,main_report,Final adjusted total cost model using Gamma GLM.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...,copied
9,notebook_05_high_cost_model_odds_ratios.png,10_final_high_cost_logistic_model_odds_ratios.png,05,main_report,Final adjusted high-cost

All main-report figures were copied successfully.


## 5. Build master summary table

This table combines the key report-summary rows from Notebooks 03, 04 and 05. 


In [6]:
def read_finding_value_table(path, section_label):
    if not path.exists():
        return pd.DataFrame(columns=["section", "finding", "value", "source_file"])

    df = pd.read_csv(path)

    # Standard format used by report summary tables
    if {"finding", "value"}.issubset(df.columns):
        out = df[["finding", "value"]].copy()
    elif {"assessment_item", "value"}.issubset(df.columns):
        out = df.rename(columns={"assessment_item": "finding"})[["finding", "value"]].copy()
    elif {"metric", "value"}.issubset(df.columns):
        out = df.rename(columns={"metric": "finding"})[["finding", "value"]].copy()
    else:
        # Fallback: convert first two columns to finding/value
        first_two = df.columns[:2]
        out = df[list(first_two)].copy()
        out.columns = ["finding", "value"]

    out.insert(0, "section", section_label)
    out["source_file"] = path.name
    return out

master_summary_parts = [
    read_finding_value_table(final_tables_dir / "02_notebook_02_cohort_size_assessment.csv", "Data readiness"),
    read_finding_value_table(final_tables_dir / "03_notebook_03_report_summary.csv", "Descriptive cohort profile"),
    read_finding_value_table(final_tables_dir / "05_notebook_04_report_summary.csv", "Utilisation and cost analysis"),
    read_finding_value_table(final_tables_dir / "09_notebook_05_report_summary.csv", "Adjusted modelling")
]

master_summary = pd.concat(master_summary_parts, ignore_index=True)
master_summary.to_csv(report_assets_dir / "master_summary_table.csv", index=False)

display(master_summary)


,section,finding,value,source_file
0,Data readiness,total_cleaned_rows,50000,02_notebook_02_cohort_size_assessment.csv
1,Data readiness,primary_cancer_rows,1218,02_notebook_02_cohort_size_assessment.csv
2,Data readiness,broader_cancer_related_rows,1377,02_notebook_02_cohort_size_assessment.csv
3,Data readiness,model_input_rows,1218,02_notebook_02_cohort_size_assessment.csv
4,Data readiness,minimum_primary_cancer_rows_for_final_modelling,500,02_notebook_02_cohort_size_assessment.csv
5,Data readiness,suitable_for_pipeline_testing,Yes,02_notebook_02_cohort_size_assessment.csv
6,Data readiness,suitable_for_final_modelling,Yes,02_notebook_02_cohort_size_assessment.csv
7,Data readiness,analysis_dataset_status,final_analysis_ready,02_notebook_02_cohort_size_assessment.csv
8,Data readiness,recommended_next_step,Proceed to final descriptive analysis and modelling.,02_notebook_02_cohort_size_assessment.csv
9,Descriptive cohort profile,Total admissions in cleaned sample,50000,03_notebook_03_report_summary.csv


## 6. Build final model effects table

This table extracts only the final report-facing model results:

1. Negative Binomial LOS model.
2. Gamma cost model.
3. Logistic high-cost model.

It deliberately excludes the intercept and avoids mixing in baseline or sensitivity models.


In [7]:
def load_model_effects(path, model_label, effect_col, lower_col, upper_col, effect_label):
    if not path.exists():
        return pd.DataFrame(columns=[
            "model", "outcome", "term", "term_label", "effect_measure",
            "estimate", "ci_lower", "ci_upper", "formatted_effect", "p_value",
            "source_file"
        ])

    df = pd.read_csv(path)
    df = df[df["term"].ne("const")].copy() if "term" in df.columns else df.copy()

    if "term_label" not in df.columns and "term" in df.columns:
        df["term_label"] = df["term"]

    out = pd.DataFrame({
        "model": model_label,
        "term": df.get("term", pd.Series(dtype=str)),
        "term_label": df.get("term_label", pd.Series(dtype=str)),
        "effect_measure": effect_label,
        "estimate": df[effect_col],
        "ci_lower": df[lower_col],
        "ci_upper": df[upper_col],
        "p_value": df.get("p_value", pd.Series([np.nan] * len(df))),
        "source_file": path.name
    })

    out["formatted_effect"] = (
        out["estimate"].map(lambda x: f"{x:.2f}") +
        " (" +
        out["ci_lower"].map(lambda x: f"{x:.2f}") +
        " to " +
        out["ci_upper"].map(lambda x: f"{x:.2f}") +
        ")"
    )

    return out

model_effects = pd.concat([
    load_model_effects(
        final_tables_dir / "11_notebook_05_los_negative_binomial_model_results_report_facing.csv",
        model_label="Negative Binomial GLM for length of stay",
        effect_col="incidence_rate_ratio",
        lower_col="incidence_rate_ratio_ci_lower",
        upper_col="incidence_rate_ratio_ci_upper",
        effect_label="Incidence rate ratio"
    ),
    load_model_effects(
        final_tables_dir / "12_notebook_05_cost_gamma_model_results_report_facing.csv",
        model_label="Gamma GLM with log link for total costs",
        effect_col="cost_ratio",
        lower_col="cost_ratio_ci_lower",
        upper_col="cost_ratio_ci_upper",
        effect_label="Cost ratio"
    ),
    load_model_effects(
        final_tables_dir / "13_notebook_05_high_cost_logistic_model_results_report_facing.csv",
        model_label="Logistic GLM for high-cost admission",
        effect_col="odds_ratio",
        lower_col="odds_ratio_ci_lower",
        upper_col="odds_ratio_ci_upper",
        effect_label="Odds ratio"
    )
], ignore_index=True)

model_effects.to_csv(report_assets_dir / "final_model_effects_summary.csv", index=False)

display(model_effects)


,model,term,term_label,effect_measure,estimate,ci_lower,ci_upper,p_value,source_file,formatted_effect
0,Negative Binomial GLM for length of stay,severity_score,Severity score,Incidence rate ratio,1.8945,1.7416,2.0609,0.0000,11_notebook_05_los_negative_binomial_model_results_report_facing.csv,1.89 (1.74 to 2.06)
1,Negative Binomial GLM for length of stay,is_emergency_admission_int,Emergency admission,Incidence rate ratio,1.0058,0.8684,1.1649,0.9385,11_notebook_05_los_negative_binomial_model_results_report_facing.csv,1.01 (0.87 to 1.16)
2,Negative Binomial GLM for length of stay,older_adult_70plus,Age 70 or older,Incidence rate ratio,1.1319,0.9753,1.3137,0.1029,11_notebook_05_los_negative_binomial_model_results_report_facing.csv,1.13 (0.98 to 1.31)
3,Negative Binomial GLM for length of stay,haematologic_cancer,Haematologic cancer,Incidence rate ratio,1.6797,1.3986,2.0172,0.0000,11_notebook_05_los_negative_binomial_model_results_report_facing.csv,1.68 (1.40 to 2.02)
4,Gamma GLM with log link for total costs,severity_score,Severity score,Cost ratio,1.6523,1.5575,1.7528,0.0000,12_notebook_05_cost_gamma_model_results_report_facing.csv,1.65 (1.56 to 1.75)
5,Gamma GLM with log link for total costs,is_emergency_admission_int,Emergency admission,Cost ratio,0.7436,0.6643,0.8323,0.0000,12_notebook_05_cost_gamma_model_results_report_facing.csv,0.74 (0.66 to 0.83)
6,Gamma GLM with log link for total costs,older_adult_70plus,Age 70 or older,Cost ratio,0.8759,0.7988,0.9606,0.0049,12_notebook_05_cost_gamma_model_results_report_facing.csv,0.88 (0.80 to 0.96)
7,Gamma GLM with log link for total costs,haematologic_cancer,Haematologic cancer,Cost ratio,1.6804,1.3829,2.0419,0.0000,12_notebook_05_cost_gamma_model_results_report_facing.csv,1.68 (1.38 to 2.04)
8,Logistic GLM for high-cost admission,severity_score,Severity score,Odds ratio,3.7899,3.0609,4.6926,0.0000,13_notebook_05_high_cost_logistic_model_results_report_facing.csv,3.79 (3.06 to 4.69)
9,Logistic GLM for high-cost admission,is_emergency_admission_int,Emergency admission,Odds ratio,0.6072,0.4346,0.8484,0.0035,13_notebook_05_high_cost_logistic_model_results_report_facing.csv,0.61 (0.43 to 0.85)


## 7. Create report figure shortlist

This table is the clean figure shortlist for the final report. It prevents accidental use of old prototype figures.


In [8]:
final_report_figure_shortlist = figure_inventory[
    (figure_inventory["role"] == "main_report") &
    (figure_inventory["status"] == "copied")
].copy()

final_report_figure_shortlist = final_report_figure_shortlist[
    ["final_file", "source_notebook", "purpose", "destination_path"]
]

final_report_figure_shortlist.to_csv(
    report_assets_dir / "final_report_figure_shortlist.csv",
    index=False
)

display(final_report_figure_shortlist)


,final_file,source_notebook,purpose,destination_path
0,01_median_total_costs_primary_cancer_vs_non_cancer.png,03,Compares median total costs between primary cancer and non-cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
1,02_median_los_primary_cancer_vs_non_cancer.png,03,Compares median length of stay between primary cancer and non-cancer admissions.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
2,03_top_primary_cancer_diagnosis_groups.png,03,Describes the distribution of primary cancer diagnosis groups.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
3,04_median_total_costs_by_severity.png,04,Shows unadjusted cost gradient by APR severity group.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
4,05_median_los_by_severity.png,04,Shows unadjusted LOS gradient by APR severity group.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
5,06_median_total_costs_by_diagnosis_group_min_n20.png,04,Diagnosis-group cost ranking using minimum n≥20 threshold.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
6,07_median_los_by_diagnosis_group_min_n20.png,04,Diagnosis-group LOS ranking using minimum n≥20 threshold.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
7,08_final_negative_binomial_los_model_irr.png,05,Final adjusted LOS model using Negative Binomial GLM.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
8,09_final_gamma_cost_model_cost_ratios.png,05,Final adjusted total cost model using Gamma GLM.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...
9,10_final_high_cost_logistic_model_odds_ratios.png,05,Final adjusted high-cost admission model using logistic GLM.,/Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outpu...


## 8. Create evidence-pack manifest and README

The manifest is machine-readable. The README is human-readable and useful for GitHub.


In [9]:
manifest = {
    "project": "Cancer-related inpatient utilisation and cost variation using SPARCS administrative discharge data",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_dir": str(project_dir),
    "final_tables_dir": str(final_tables_dir),
    "final_figures_main_dir": str(final_figures_main_dir),
    "final_figures_diagnostics_dir": str(final_figures_diagnostics_dir),
    "report_assets_dir": str(report_assets_dir),
    "main_report_tables_copied": int(((table_inventory["role"] == "main_report") & (table_inventory["status"] == "copied")).sum()),
    "main_report_tables_missing": int(((table_inventory["role"] == "main_report") & (table_inventory["status"] == "missing")).sum()),
    "main_report_figures_copied": int(((figure_inventory["role"] == "main_report") & (figure_inventory["status"] == "copied")).sum()),
    "main_report_figures_missing": int(((figure_inventory["role"] == "main_report") & (figure_inventory["status"] == "missing")).sum()),
    "final_modelling_strategy": {
        "length_of_stay": "Negative Binomial GLM",
        "total_costs": "Gamma GLM with log link",
        "high_cost_admission": "Logistic GLM"
    },
    "do_not_use": [
        "notebook_05_cost_model_cost_ratios.png",
        "notebook_05_los_model_incidence_rate_ratios.png"
    ]
}

with open(report_assets_dir / "final_evidence_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

readme_text = f'''# Final Evidence Pack

## Project

Cancer-related inpatient utilisation and cost variation using SPARCS administrative discharge data.

## Created

{manifest["created_at"]}

## Folder structure

- `outputs/final_tables/`: selected tables for the final report and evidence pack.
- `outputs/final_figures/main_report/`: selected figures for the main report.
- `outputs/final_figures/diagnostics_appendix/`: diagnostic and sensitivity figures.
- `outputs/report_assets/`: inventories, master summary and final model effects summary.

## Final modelling strategy

The final report should use:

1. Negative Binomial GLM for length of stay.
2. Gamma GLM with log link for total inpatient costs.
3. Logistic GLM for high-cost admission status.

The Poisson LOS model and log-linear OLS cost model are retained only as diagnostic or sensitivity outputs.

## Main report model figures

- `08_final_negative_binomial_los_model_irr.png`
- `09_final_gamma_cost_model_cost_ratios.png`
- `10_final_high_cost_logistic_model_odds_ratios.png`

## Do not use

The following old duplicate figures should not be used in the report:

- `notebook_05_cost_model_cost_ratios.png`
- `notebook_05_los_model_incidence_rate_ratios.png`

## Key report assets

- `master_summary_table.csv`
- `final_model_effects_summary.csv`
- `table_inventory.csv`
- `figure_inventory.csv`
- `final_report_figure_shortlist.csv`
- `final_evidence_manifest.json`
'''

with open(report_assets_dir / "README_final_evidence_pack.md", "w") as f:
    f.write(readme_text)

print("Created manifest:", report_assets_dir / "final_evidence_manifest.json")
print("Created README:", report_assets_dir / "README_final_evidence_pack.md")


Created manifest: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/report_assets/final_evidence_manifest.json
Created README: /Users/marissa/Desktop/portfolio/oncology-health-economics-portfolio/project_01_sparcs_cancer_utilisation_cost/outputs/report_assets/README_final_evidence_pack.md


## 9. Final quality checks

This is the gatekeeper cell. If it flags missing main-report assets, do not write the final report yet.


In [10]:
quality_checks = pd.DataFrame({
    "check": [
        "Main-report tables missing",
        "Main-report figures missing",
        "Deprecated cost figure excluded",
        "Deprecated LOS figure excluded",
        "Final model effects table rows",
        "Master summary rows"
    ],
    "value": [
        int(((table_inventory["role"] == "main_report") & (table_inventory["status"] == "missing")).sum()),
        int(((figure_inventory["role"] == "main_report") & (figure_inventory["status"] == "missing")).sum()),
        "Yes" if "notebook_05_cost_model_cost_ratios.png" in figure_inventory.loc[figure_inventory["role"].eq("exclude"), "source_file"].values else "No",
        "Yes" if "notebook_05_los_model_incidence_rate_ratios.png" in figure_inventory.loc[figure_inventory["role"].eq("exclude"), "source_file"].values else "No",
        int(len(model_effects)),
        int(len(master_summary))
    ]
})

quality_checks["status"] = np.where(
    quality_checks["check"].isin(["Main-report tables missing", "Main-report figures missing"]) &
    quality_checks["value"].astype(str).ne("0"),
    "Needs attention",
    "OK"
)

quality_checks.to_csv(report_assets_dir / "final_quality_checks.csv", index=False)

display(quality_checks)

if (quality_checks["status"] == "Needs attention").any():
    print("Do not write the final report yet. Fix the missing assets first.")
else:
    print("Final evidence pack is ready for report writing.")


,check,value,status
0,Main-report tables missing,0,OK
1,Main-report figures missing,0,OK
2,Deprecated cost figure excluded,Yes,OK
3,Deprecated LOS figure excluded,Yes,OK
4,Final model effects table rows,11,OK
5,Master summary rows,39,OK


Final evidence pack is ready for report writing.


## 10. Notebook conclusion



The final report should now be based on:

1. The master summary table.
2. The final model effects table.
3. The selected main-report figure shortlist.
4. The final-report tables copied into `outputs/final_tables/`.

